# Graph Animation Examples

Each example below builds a `Schedule` directly in its own code cell.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    CurveStyleTransition,
    DrawTransition,
    EraseTransition,
    FillBetweenArea,
    FillBetweenTransition,
    JitterTransition,
    MoveFillBetweenTransition,
    MoveTransition,
    ParallelTransition,
    Schedule,
    StressTransition,
)


In [ ]:
# Example 1: draw two curves concurrently
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))
y_arch = np.clip(0.2 + 0.7 * (1.0 - np.square(2.0 * x - 1.0)), 0.0, 1.0)

schedule = Schedule().add(
    ParallelTransition(
        (
            DrawTransition(Curve("wave_a", x, y_wave, color="#0f766e", linewidth=3.0)),
            DrawTransition(
                Curve("arch_a", x, y_arch, color="#2563eb", linewidth=2.5, linestyle="--")
            ),
        )
    ),
    duration=2.0,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = schedule.build_animation(fig=fig, ax=ax, fps=30, title="Concurrent draw")
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 2: draw a curve and its fill_between at the same time
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))

schedule = Schedule().add(
    ParallelTransition(
        (
            DrawTransition(Curve("wave_b", x, y_wave, color="#7c3aed", linewidth=3.0)),
            FillBetweenTransition(
                FillBetweenArea(
                    "wave_b_fill",
                    x,
                    y_wave,
                    0.0,
                    color="#c4b5fd",
                    alpha=0.45,
                    linewidth=0.0,
                )
            ),
        )
    ),
    duration=1.8,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = schedule.build_animation(fig=fig, ax=ax, fps=30, title="Curve and fill together")
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 3: sudden plot and sudden style change with zero-duration transitions
x = np.linspace(0.0, 1.0, 250)
y_arch = np.clip(0.2 + 0.7 * (1.0 - np.square(2.0 * x - 1.0)), 0.0, 1.0)

schedule = Schedule()
schedule.add(
    DrawTransition(
        Curve("snap", x, y_arch, color="#0ea5e9", linewidth=2.0, linestyle="solid"),
        show_pointer=False,
    ),
    duration=0.0,
)
schedule.add(
    CurveStyleTransition("snap", color="#16a34a", linestyle="--", linewidth=3.0),
    duration=0.0,
)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = schedule.build_animation(fig=fig, ax=ax, fps=30, title="Instant plot and style switch")
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 4: slowly change alpha and color
x = np.linspace(0.0, 1.0, 250)
y_ramp = 0.12 + 0.72 * np.power(x, 1.1)

schedule = Schedule()
schedule.add(
    DrawTransition(
        Curve("fade", x, y_ramp, color="#94a3b8", alpha=0.0, linewidth=3.0),
        show_pointer=False,
    ),
    duration=0.0,
)
schedule.add(CurveStyleTransition("fade", color="#0f766e", alpha=1.0), duration=1.5)
schedule.add(CurveStyleTransition("fade", color="#f59e0b", alpha=0.15), duration=1.0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = schedule.build_animation(fig=fig, ax=ax, fps=30, title="Color and alpha interpolation")
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 5: insert a break where nothing happens
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))

schedule = Schedule()
schedule.add(DrawTransition(Curve("pause_demo", x, y_wave, color="#be123c", linewidth=3.0)), duration=1.5)
schedule.add_break(0.9)
schedule.add(StressTransition("pause_demo", glow_color="#fb7185", glow_width=10.0), duration=0.8)
schedule.add(EraseTransition("pause_demo"), duration=1.2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = schedule.build_animation(fig=fig, ax=ax, fps=30, title="Break in the schedule")
plt.close(fig)
HTML(anim.to_jshtml())


In [ ]:
# Example 6: full story with draw, fill, pause, stress, jitter, move, style shift, and erase
x = np.linspace(0.0, 1.0, 250)
y_wave = np.abs(np.sin(x))
y_square = np.square(y_wave)

schedule = Schedule()
schedule.add(
    ParallelTransition(
        (
            DrawTransition(Curve("story_wave", x, y_wave, color="#0f766e", linewidth=3.0)),
            FillBetweenTransition(
                FillBetweenArea(
                    "story_fill",
                    x,
                    y_wave,
                    0.0,
                    color="#99f6e4",
                    alpha=0.35,
                    linewidth=0.0,
                )
            ),
        )
    ),
    duration=1.8,
)
schedule.pause(0.5)
schedule.add(StressTransition("story_wave", glow_color="#14b8a6", glow_width=11.0), duration=0.8)
schedule.add(JitterTransition("story_wave", y_amplitude=0.03, cycles=10.0, seed=7), duration=0.8)
schedule.add(
    ParallelTransition(
        (
            MoveTransition("story_wave", x_prime=x, y_prime=y_square, color="#0f766e", linewidth=3.0),
            MoveFillBetweenTransition(
                "story_fill",
                x_prime=x,
                y1_prime=y_square,
                y2_prime=np.zeros_like(y_square),
                color="#fcd34d",
                alpha=0.35,
            ),
        )
    ),
    duration=1.8,
)
schedule.add(CurveStyleTransition("story_wave", color="#ea580c", alpha=0.8, linewidth=4.0), duration=1.0)
schedule.add(EraseTransition("story_wave"), duration=1.2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.grid(True, alpha=0.2)
anim = schedule.build_animation(fig=fig, ax=ax, fps=30, title="Combined transitions")
plt.close(fig)
HTML(anim.to_jshtml())
